# 6. Taxi + Zone Analysis

This notebook combines taxi trip records with the taxi zone reference
dataset.

The zone dataset provides geographic and administrative information for
the pickup and drop-off locations.

The resulting dataset will support demand analysis, hotspot analysis,
origin-destination analysis, and zone-level visualisation.

In [2]:
# ============================================================
# 6.1 Load Zone Dataset
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()

# If running from the notebook directory, move to project root
while PROJECT_ROOT.name != "UrbanFlow_AI" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

ZONE_DIR = PROJECT_ROOT / "data" / "raw" / "zone"

zone_files = list(ZONE_DIR.glob("*.csv"))

print("Zone files found:")
for file in zone_files:
    print(file.name)

Zone files found:
Urban_Flow_Analytics_Zone_Dataset.csv


In [3]:
# Read the zone reference file
zone_file = zone_files[0]

zones = pd.read_csv(zone_file)

print(f"Rows: {len(zones):,}")
print(f"Columns: {len(zones.columns)}")

zones.head()

Rows: 265
Columns: 4


,loc_id,borough_name,zone_name,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [4]:
# Inspect structure
zones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   loc_id        265 non-null    int64 
 1   borough_name  264 non-null    object
 2   zone_name     264 non-null    object
 3   service_zone  263 non-null    object
dtypes: int64(1), object(3)
memory usage: 8.4+ KB


In [5]:
# Check missing values and duplicate zone IDs
print("Missing values:")
display(zones.isna().sum())

print("\nDuplicate loc_id values:")
print(zones["loc_id"].duplicated().sum())

print("\nUnique loc_id values:")
print(zones["loc_id"].nunique())

Missing values:


loc_id          0
borough_name    1
zone_name       1
service_zone    2
dtype: int64


Duplicate loc_id values:
0

Unique loc_id values:
265


## 6.2 Prepare the Zone Lookup

The zone reference table contains 265 unique location IDs.

We will create two lookup tables:
- one for pickup zones
- 
- one for drop-off zones

The columns are renamed to clearly distinguish pickup and drop-off
information after joining with the taxi records.

In [7]:
# ============================================================
# 6.2 Prepare Pickup and Drop-off Zone Lookups
# ============================================================

pickup_zones = zones.rename(columns={
    "loc_id": "origin_loc_id",
    "borough_name": "pickup_borough",
    "zone_name": "pickup_zone",
    "service_zone": "pickup_service_zone"
}).copy()

dropoff_zones = zones.rename(columns={
    "loc_id": "dest_loc_id",
    "borough_name": "dropoff_borough",
    "zone_name": "dropoff_zone",
    "service_zone": "dropoff_service_zone"
}).copy()

print("Pickup lookup:")
display(pickup_zones.head())

print("\nDrop-off lookup:")
display(dropoff_zones.head())

Pickup lookup:


,origin_loc_id,pickup_borough,pickup_zone,pickup_service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone



Drop-off lookup:


,dest_loc_id,dropoff_borough,dropoff_zone,dropoff_service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


## 6.3 Test Taxi-Zone Join

The taxi dataset stores only numeric origin and destination location IDs.

These IDs are joined with the zone reference table to obtain meaningful
geographic information such as borough, zone name, and service zone.

The join is performed as a left join so that no taxi trip is silently
removed because of a missing zone reference.

In [9]:
# ============================================================
# 6.3 Test Taxi-Zone Join
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Find the project root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

while (
    PROJECT_ROOT.name != "UrbanFlow_AI"
    and PROJECT_ROOT.parent != PROJECT_ROOT
):
    PROJECT_ROOT = PROJECT_ROOT.parent

# ------------------------------------------------------------
# Define paths locally for this notebook
# ------------------------------------------------------------

PROCESSED_TAXI_DIR = (
    PROJECT_ROOT / "data" / "processed" / "taxi_clean"
)

# ------------------------------------------------------------
# Locate the April processed file
# ------------------------------------------------------------

april_file = (
    PROCESSED_TAXI_DIR
    / "Urban_Flow_Analytics_Taxi_Dataset_2025-04_processed.parquet"
)

print("File exists:", april_file.exists())
print("File:", april_file)

# ------------------------------------------------------------
# Load a small sample
# ------------------------------------------------------------

zone_test = pd.read_parquet(
    april_file
).head(10_000).copy()

print(f"Test records: {len(zone_test):,}")

# ------------------------------------------------------------
# Join pickup-zone information
# ------------------------------------------------------------

zone_test = zone_test.merge(
    pickup_zones,
    on="origin_loc_id",
    how="left"
)

# ------------------------------------------------------------
# Join drop-off-zone information
# ------------------------------------------------------------

zone_test = zone_test.merge(
    dropoff_zones,
    on="dest_loc_id",
    how="left"
)

# ------------------------------------------------------------
# Check zone matching
# ------------------------------------------------------------

pickup_failures = zone_test["pickup_zone"].isna().sum()
dropoff_failures = zone_test["dropoff_zone"].isna().sum()

print(f"Pickup zone match failures: {pickup_failures:,}")
print(f"Drop-off zone match failures: {dropoff_failures:,}")

# Show the joined result
display(
    zone_test[
        [
            "origin_loc_id",
            "pickup_borough",
            "pickup_zone",
            "dest_loc_id",
            "dropoff_borough",
            "dropoff_zone"
        ]
    ].head(10)
)

File exists: True
File: C:\Users\arudk\Downloads\UrbanFlow_AI\data\processed\taxi_clean\Urban_Flow_Analytics_Taxi_Dataset_2025-04_processed.parquet
Test records: 10,000
Pickup zone match failures: 32
Drop-off zone match failures: 34


,origin_loc_id,pickup_borough,pickup_zone,dest_loc_id,dropoff_borough,dropoff_zone
0,138,Queens,LaGuardia Airport,230,Manhattan,Times Sq/Theatre District
1,138,Queens,LaGuardia Airport,92,Queens,Flushing
2,132,Queens,JFK Airport,130,Queens,Jamaica
3,79,Manhattan,East Village,4,Manhattan,Alphabet City
4,161,Manhattan,Midtown Center,229,Manhattan,Sutton Place/Turtle Bay North
5,233,Manhattan,UN/Turtle Bay South,164,Manhattan,Midtown South
6,138,Queens,LaGuardia Airport,140,Manhattan,Lenox Hill East
7,138,Queens,LaGuardia Airport,116,Manhattan,Hamilton Heights
8,239,Manhattan,Upper West Side South,238,Manhattan,Upper West Side North
9,132,Queens,JFK Airport,162,Manhattan,Midtown East


## 6.4 Build Zone-Level Demand and OD Statistics

Instead of creating another very large row-level joined dataset, we aggregate
the taxi records directly while processing each monthly file.

This produces compact zone-level and origin-destination datasets that are
much easier to analyse and use for forecasting.

Pickup and drop-off zone information is obtained through the zone reference
table.

In [19]:
# ============================================================
# 6.4 Taxi + Zone Aggregation
# Self-contained version
# ============================================================

from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Find project root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

while (
    PROJECT_ROOT.name != "UrbanFlow_AI"
    and PROJECT_ROOT.parent != PROJECT_ROOT
):
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_TAXI_DIR = PROJECT_ROOT / "data" / "raw" / "taxi"
ZONE_DIR = PROJECT_ROOT / "data" / "raw" / "zone"

OUTPUT_DIR = (
    PROJECT_ROOT / "data" / "processed" / "taxi_zone"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

# ------------------------------------------------------------
# 2. Load zone reference
# ------------------------------------------------------------

zone_file = list(ZONE_DIR.glob("*.csv"))[0]
zones = pd.read_csv(zone_file)

# Create lookup dictionaries
borough_lookup = (
    zones.set_index("loc_id")["borough_name"].to_dict()
)

zone_name_lookup = (
    zones.set_index("loc_id")["zone_name"].to_dict()
)

# ------------------------------------------------------------
# 3. Aggregation containers
# ------------------------------------------------------------

hourly_demand = defaultdict(int)
pickup_zone_counts = defaultdict(int)
dropoff_zone_counts = defaultdict(int)
od_counts = defaultdict(int)

am_pickup_counts = defaultdict(int)
pm_pickup_counts = defaultdict(int)

# ------------------------------------------------------------
# 4. Process each monthly taxi file
# ------------------------------------------------------------

taxi_files = sorted(RAW_TAXI_DIR.glob("*.csv"))

print(f"Taxi files found: {len(taxi_files)}")

for file_path in taxi_files:

    print(f"\nProcessing: {file_path.name}")

    for chunk in pd.read_csv(
        file_path,
        usecols=[
            "pickup_timestamp",
            "origin_loc_id",
            "dest_loc_id"
        ],
        chunksize=500_000
    ):

        # Convert timestamp
        chunk["pickup_timestamp"] = pd.to_datetime(
            chunk["pickup_timestamp"],
            errors="coerce"
        )

        # Remove unusable records
        chunk = chunk.dropna(
            subset=[
                "pickup_timestamp",
                "origin_loc_id",
                "dest_loc_id"
            ]
        )

        # ----------------------------------------------------
        # Time features
        # ----------------------------------------------------

        chunk["pickup_hour"] = (
            chunk["pickup_timestamp"].dt.hour
        )

        chunk["pickup_date"] = (
            chunk["pickup_timestamp"].dt.date
        )

        # ----------------------------------------------------
        # Hourly demand
        # ----------------------------------------------------

        for key, count in (
            chunk.groupby(
                ["pickup_date", "pickup_hour"]
            ).size().items()
        ):
            hourly_demand[key] += int(count)

        # ----------------------------------------------------
        # Pickup zones
        # ----------------------------------------------------

        for zone_id, count in (
            chunk["origin_loc_id"].value_counts().items()
        ):
            pickup_zone_counts[int(zone_id)] += int(count)

        # ----------------------------------------------------
        # Drop-off zones
        # ----------------------------------------------------

        for zone_id, count in (
            chunk["dest_loc_id"].value_counts().items()
        ):
            dropoff_zone_counts[int(zone_id)] += int(count)

        # ----------------------------------------------------
        # OD flows
        # ----------------------------------------------------

        for key, count in (
            chunk.groupby(
                ["origin_loc_id", "dest_loc_id"]
            ).size().items()
        ):
            od_counts[
                (int(key[0]), int(key[1]))
            ] += int(count)

        # ----------------------------------------------------
        # AM hotspots: 06:00-11:59
        # PM hotspots: 12:00-17:59
        # ----------------------------------------------------

        for zone_id, count in (
            chunk.loc[
                chunk["pickup_hour"].between(6, 11),
                "origin_loc_id"
            ].value_counts().items()
        ):
            am_pickup_counts[int(zone_id)] += int(count)

        for zone_id, count in (
            chunk.loc[
                chunk["pickup_hour"].between(12, 17),
                "origin_loc_id"
            ].value_counts().items()
        ):
            pm_pickup_counts[int(zone_id)] += int(count)

# ------------------------------------------------------------
# 5. Create hourly demand table
# ------------------------------------------------------------

hourly_demand_df = pd.DataFrame(
    [
        {
            "pickup_date": date,
            "pickup_hour": hour,
            "trip_count": count
        }
        for (date, hour), count in hourly_demand.items()
    ]
)

hourly_demand_df["pickup_date"] = pd.to_datetime(
    hourly_demand_df["pickup_date"]
)

hourly_demand_df = hourly_demand_df.sort_values(
    ["pickup_date", "pickup_hour"]
)

# ------------------------------------------------------------
# 6. Pickup zone statistics
# ------------------------------------------------------------

pickup_zone_df = pd.DataFrame({
    "loc_id": list(pickup_zone_counts.keys()),
    "pickup_trip_count": list(pickup_zone_counts.values())
})

pickup_zone_df["borough_name"] = (
    pickup_zone_df["loc_id"].map(borough_lookup)
)

pickup_zone_df["zone_name"] = (
    pickup_zone_df["loc_id"].map(zone_name_lookup)
)

# ------------------------------------------------------------
# 7. Drop-off zone statistics
# ------------------------------------------------------------

dropoff_zone_df = pd.DataFrame({
    "loc_id": list(dropoff_zone_counts.keys()),
    "dropoff_trip_count": list(dropoff_zone_counts.values())
})

dropoff_zone_df["borough_name"] = (
    dropoff_zone_df["loc_id"].map(borough_lookup)
)

dropoff_zone_df["zone_name"] = (
    dropoff_zone_df["loc_id"].map(zone_name_lookup)
)

# ------------------------------------------------------------
# 8. OD flow table
# ------------------------------------------------------------

od_df = pd.DataFrame(
    [
        {
            "origin_loc_id": origin,
            "dest_loc_id": destination,
            "trip_count": count
        }
        for (origin, destination), count in od_counts.items()
    ]
)

od_df["origin_zone"] = (
    od_df["origin_loc_id"].map(zone_name_lookup)
)

od_df["destination_zone"] = (
    od_df["dest_loc_id"].map(zone_name_lookup)
)

# ------------------------------------------------------------
# 9. AM / PM hotspot table
# ------------------------------------------------------------

all_hotspot_zones = sorted(
    set(am_pickup_counts) |
    set(pm_pickup_counts)
)

hotspot_df = pd.DataFrame({
    "loc_id": all_hotspot_zones
})

hotspot_df["am_trip_count"] = (
    hotspot_df["loc_id"]
    .map(am_pickup_counts)
    .fillna(0)
    .astype(int)
)

hotspot_df["pm_trip_count"] = (
    hotspot_df["loc_id"]
    .map(pm_pickup_counts)
    .fillna(0)
    .astype(int)
)

hotspot_df["borough_name"] = (
    hotspot_df["loc_id"].map(borough_lookup)
)

hotspot_df["zone_name"] = (
    hotspot_df["loc_id"].map(zone_name_lookup)
)

# ------------------------------------------------------------
# 10. Save compact datasets
# ------------------------------------------------------------

hourly_demand_df.to_parquet(
    OUTPUT_DIR / "hourly_demand.parquet",
    index=False
)

pickup_zone_df.to_csv(
    OUTPUT_DIR / "pickup_zone_statistics.csv",
    index=False
)

dropoff_zone_df.to_csv(
    OUTPUT_DIR / "dropoff_zone_statistics.csv",
    index=False
)

od_df.to_csv(
    OUTPUT_DIR / "od_flow_statistics.csv",
    index=False
)

hotspot_df.to_csv(
    OUTPUT_DIR / "am_pm_hotspots.csv",
    index=False
)

print("\n" + "=" * 60)
print("AGGREGATION COMPLETE")
print("=" * 60)

print(f"Hourly records: {len(hourly_demand_df):,}")
print(f"Pickup zones: {len(pickup_zone_df):,}")
print(f"Drop-off zones: {len(dropoff_zone_df):,}")
print(f"OD pairs: {len(od_df):,}")
print(f"Hotspot zones: {len(hotspot_df):,}")

Project root: C:\Users\arudk\Downloads\UrbanFlow_AI
Taxi files found: 12

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv

AGGREGATION COMPLETE
Hourly records: 8,766
Pickup zones: 262
Drop-off zones: 263
OD pairs: 55,933
Hotspot zones: 262


In [21]:
# ============================================================
# 6.4.1 Quick Results
# ============================================================

print("TOP 15 PICKUP ZONES")
display(
    pickup_zone_df
    .sort_values("pickup_trip_count", ascending=False)
    .head(15)
)

print("\nTOP 15 OD FLOWS")
display(
    od_df
    .sort_values("trip_count", ascending=False)
    .head(15)
)

print("\nTOP 10 AM HOTSPOTS")
display(
    hotspot_df
    .sort_values("am_trip_count", ascending=False)
    .head(10)
)

print("\nTOP 10 PM HOTSPOTS")
display(
    hotspot_df
    .sort_values("pm_trip_count", ascending=False)
    .head(10)
)

TOP 15 PICKUP ZONES


,loc_id,pickup_trip_count,borough_name,zone_name
0,237,2113429,Manhattan,Upper East Side South
3,132,2059989,Queens,JFK Airport
1,161,2023458,Manhattan,Midtown Center
2,236,1867732,Manhattan,Upper East Side North
5,186,1508078,Manhattan,Penn Station/Madison Sq West
4,162,1477633,Manhattan,Midtown East
6,230,1450984,Manhattan,Times Sq/Theatre District
8,142,1371620,Manhattan,Lincoln Square East
10,170,1273860,Manhattan,Murray Hill
11,234,1268407,Manhattan,Union Sq



TOP 15 OD FLOWS


,origin_loc_id,dest_loc_id,trip_count,origin_zone,destination_zone
12071,237,236,301532,Upper East Side South,Upper East Side North
11910,236,237,258572,Upper East Side North,Upper East Side South
12072,237,237,214271,Upper East Side South,Upper East Side South
11909,236,236,196494,Upper East Side North,Upper East Side North
8188,161,237,140579,Midtown Center,Upper East Side South
12021,237,161,132170,Upper East Side South,Midtown Center
8187,161,236,112593,Midtown Center,Upper East Side North
5949,132,132,105773,JFK Airport,JFK Airport
12022,237,162,105640,Upper East Side South,Midtown East
7060,142,239,104177,Lincoln Square East,Upper West Side South



TOP 10 AM HOTSPOTS


,loc_id,am_trip_count,pm_trip_count,borough_name,zone_name
232,236,546086,835406,Manhattan,Upper East Side North
233,237,471141,933326,Manhattan,Upper East Side South
182,186,398321,504959,Manhattan,Penn Station/Madison Sq West
158,162,323452,551897,Manhattan,Midtown East
128,132,318219,718535,Queens,JFK Airport
157,161,315919,804979,Manhattan,Midtown Center
235,239,308150,491634,Manhattan,Upper West Side South
166,170,287499,472451,Manhattan,Murray Hill
138,142,280152,489304,Manhattan,Lincoln Square East
137,141,278969,384463,Manhattan,Lenox Hill West



TOP 10 PM HOTSPOTS


,loc_id,am_trip_count,pm_trip_count,borough_name,zone_name
233,237,471141,933326,Manhattan,Upper East Side South
232,236,546086,835406,Manhattan,Upper East Side North
157,161,315919,804979,Manhattan,Midtown Center
128,132,318219,718535,Queens,JFK Airport
158,162,323452,551897,Manhattan,Midtown East
182,186,398321,504959,Manhattan,Penn Station/Madison Sq West
134,138,238671,500901,Queens,LaGuardia Airport
235,239,308150,491634,Manhattan,Upper West Side South
138,142,280152,489304,Manhattan,Lincoln Square East
166,170,287499,472451,Manhattan,Murray Hill
